# RL Token Autoencoder — Embedding Analysis

Visualize what the RLT autoencoder reconstructs well vs poorly.

- **Inputs:** dumped Pi0.5-CoT prefix embeddings `z_{1:M}` in `rl_token_embeddings/traffic_light_v0/{train,val}`
- **Model:** a trained AE checkpoint in `rl_token_ae/<run>/ae_ckpt.pt`
- **Focus:** separate **low-loss** vs **high-loss** samples and see *where* (which tokens / segments) and *what* (latent structure, sequence length, actions) drives the difference.

Run from the repo root on the machine that has the data + checkpoints.

## 1. Setup, paths, and load the model

In [ ]:
import glob, pathlib, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# --- Paths (edit RUN to the checkpoint you want to analyze) ---
REPO = pathlib.Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

EMB_DIR   = REPO / "rl_token_embeddings" / "traffic_light_v0"
VAL_DIR   = EMB_DIR / "val"
TRAIN_DIR = EMB_DIR / "train"

RUN  = "traffic_light_v0"                       # <-- subdir under rl_token_ae/
CKPT = REPO / "rl_token_ae" / RUN / "ae_ckpt.pt"

MAX_SAMPLES = None                              # cap loaded val samples for speed, or None
device = "cuda" if torch.cuda.is_available() else "cpu"

print("repo :", REPO)
print("ckpt :", CKPT, "| exists:", CKPT.exists())
print("val  :", VAL_DIR, "| exists:", VAL_DIR.exists())
print("device:", device)

In [ ]:
from openpi.models.rl_token import RLTokenAEConfig, RLTokenAutoencoder

ckpt = torch.load(CKPT, map_location=device)
cfg = RLTokenAEConfig(**ckpt["cfg"])
model = RLTokenAutoencoder(cfg).to(device).eval()
model.load_state_dict(ckpt["model"])
print("cfg:", cfg)
print("params: %.2fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

## 2. Load the dumped embeddings

Each `.npz` shard holds `prefix_out` (the per-token embeddings `z_{1:M}`), `prefix_mask`, `actions`, and the segment sizes. The compacted layout is `[vision | prompt | reasoning | subtask | fast]`.

In [ ]:
# Optional per-row raw inputs (present only if the shards were dumped with --save-metadata).
META_KEYS = ["tokenized_prompt", "tokenized_prompt_mask", "tokenized_reasoning", "tokenized_reasoning_mask",
             "tokenized_subtask", "tokenized_subtask_mask", "tokenized_fast", "tokenized_fast_mask", "base_image"]

def load_shards(shard_dir, max_samples=None):
    paths = sorted(glob.glob(f"{shard_dir}/shard_*.npz"))
    assert paths, f"no shards at {shard_dir}"
    zs, ms, acts, sizes = [], [], [], None
    metas = {k: [] for k in META_KEYS}
    for p in tqdm(paths, desc=f"load {pathlib.Path(shard_dir).name}"):
        a = np.load(p)
        zs.append(a["prefix_out"]); ms.append(a["prefix_mask"]); acts.append(a["actions"])
        for k in META_KEYS:
            if k in a.files:
                metas[k].append(a[k])
        if sizes is None:
            sizes = {k: int(a[k]) for k in ["tokens_per_cam", "n_prompt", "n_reasoning", "n_subtask", "n_fast"]}
    z = np.concatenate(zs); m = np.concatenate(ms); act = np.concatenate(acts)
    meta = {k: np.concatenate(v) for k, v in metas.items() if v}
    if max_samples:
        z, m, act = z[:max_samples], m[:max_samples], act[:max_samples]
        meta = {k: v[:max_samples] for k, v in meta.items()}
    return z, m, act, sizes, meta

# Point at TRAIN_DIR for the training-set analysis (switch to VAL_DIR if desired).
z, m, act, sizes, meta = load_shards(str(TRAIN_DIR), MAX_SAMPLES)
print("z:", z.shape, z.dtype, "| mask density: %.3f" % m.mean(), "| actions:", act.shape)
print("metadata available:", list(meta.keys()) or "(none -- re-dump with --save-metadata)")

# Segment layout of the compacted dump (see scripts/dump_rl_token_embeddings.py).
seg_lens = [("vision", sizes["tokens_per_cam"]), ("prompt", sizes["n_prompt"]),
            ("reasoning", sizes["n_reasoning"]), ("subtask", sizes["n_subtask"]), ("fast", sizes["n_fast"])]
segments, s = [], 0
for name, ln in seg_lens:
    segments.append((name, s, s + ln)); s += ln
assert s == z.shape[1], (s, z.shape[1])
print("segments:", segments)

## 3. Run the AE and collect per-sample metrics

For each sample we compute the masked-mean L2 reconstruction loss, the RL token `z_rl`, and the masked per-token L2.

In [ ]:
@torch.no_grad()
def run_metrics(z, m, bs=64):
    loss, zrl, ptl = [], [], []
    for i in tqdm(range(0, z.shape[0], bs), desc="infer"):
        zb = torch.from_numpy(z[i:i + bs]).to(device).float()
        mb = torch.from_numpy(m[i:i + bs]).to(device)
        out = model(zb, mb)
        mf = mb.float()
        per = out["per_token_l2"]                          # (b, M) unmasked
        ls = (per * mf).sum(1) / mf.sum(1).clamp(min=1)    # per-sample masked-mean L2
        loss.append(ls.cpu().numpy())
        zrl.append(out["z_rl"].float().cpu().numpy())
        ptl.append((per * mf).cpu().numpy())               # masked per-token L2
    return np.concatenate(loss), np.concatenate(zrl), np.concatenate(ptl)

loss, zrl, ptl = run_metrics(z, m)
print("per-sample loss: mean=%.2f median=%.2f min=%.2f max=%.2f"
      % (loss.mean(), np.median(loss), loss.min(), loss.max()))

## 4. Reconstruction-loss distribution

Histogram of per-sample loss, plus the indices of the lowest- and highest-loss samples for later inspection.

In [ ]:
k = 8
order = np.argsort(loss)
low_idx, high_idx = order[:k], order[-k:][::-1]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(loss, bins=60, color="steelblue")
ax[0].set_title("per-sample recon loss"); ax[0].set_xlabel("masked L2")
ax[1].hist(np.log10(loss + 1e-9), bins=60, color="indianred")
ax[1].set_title("log10 loss"); ax[1].set_xlabel("log10 L2")
plt.tight_layout(); plt.show()

print("lowest-loss idx :", low_idx, "\n  vals:", np.round(loss[low_idx], 2))
print("highest-loss idx:", high_idx, "\n  vals:", np.round(loss[high_idx], 2))

## 5. Where does the loss live? (per segment / per token)

Which parts of the prefix (vision / prompt / reasoning / subtask / fast) reconstruct well, and how the per-token profile differs between low- and high-loss samples.

In [ ]:
# Per-segment masked-mean L2 (averaged over all valid tokens in that segment).
seg_loss = {name: ptl[:, a:b].sum() / max(m[:, a:b].sum(), 1) for name, a, b in segments}
plt.figure(figsize=(7, 4))
plt.bar(list(seg_loss.keys()), list(seg_loss.values()), color="slateblue")
plt.ylabel("mean masked L2 / token"); plt.title("reconstruction loss by segment"); plt.show()

# Per-token mean across samples (valid tokens only): all vs low-loss 10% vs high-loss 10%.
qlo, qhi = np.quantile(loss, 0.1), np.quantile(loss, 0.9)
g_lo, g_hi = loss <= qlo, loss >= qhi
def tok_mean(grp):
    return ptl[grp].sum(0) / np.clip(m[grp].sum(0), 1, None)
tm_all, tm_lo, tm_hi = tok_mean(np.ones(len(loss), bool)), tok_mean(g_lo), tok_mean(g_hi)

plt.figure(figsize=(14, 4))
plt.plot(tm_all, label="all", lw=1, color="gray")
plt.plot(tm_lo, label="low-loss 10%", lw=1, color="green")
plt.plot(tm_hi, label="high-loss 10%", lw=1, color="red")
ymax = plt.ylim()[1]
for name, a, b in segments:
    plt.axvspan(a, b, alpha=0.05, color="blue")
    plt.text((a + b) / 2, ymax * 0.92, name, ha="center", fontsize=8)
plt.xlabel("token position"); plt.ylabel("mean L2"); plt.legend()
plt.title("per-token reconstruction loss"); plt.show()

## 6. RL-token latent space

Project `z_rl` to 2D (PCA, optional t-SNE) colored by reconstruction loss. Are high-loss samples outliers / clustered in latent space?

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(zrl)
emb2 = pca.transform(zrl)
c = np.log10(loss + 1e-9)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sc = ax[0].scatter(emb2[:, 0], emb2[:, 1], c=c, cmap="viridis", s=8)
ax[0].set_title("PCA of z_rl (color = log10 loss)")
ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2"); plt.colorbar(sc, ax=ax[0])
print("PCA explained var:", np.round(pca.explained_variance_ratio_, 3))

# Optional t-SNE (slower); safe to skip.
try:
    from sklearn.manifold import TSNE
    ts = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(zrl)
    sc2 = ax[1].scatter(ts[:, 0], ts[:, 1], c=c, cmap="viridis", s=8)
    ax[1].set_title("t-SNE of z_rl"); plt.colorbar(sc2, ax=ax[1])
except Exception as e:
    ax[1].set_title(f"t-SNE skipped: {e}")
plt.tight_layout(); plt.show()

## 7. Inspect individual low- vs high-loss samples

Reconstruct one low-loss and one high-loss sample; show the abs-error heatmap and per-token loss, plus segment lengths and action magnitude.

In [ ]:
@torch.no_grad()
def reconstruct(idx):
    zb = torch.from_numpy(z[idx:idx + 1]).to(device).float()
    mb = torch.from_numpy(m[idx:idx + 1]).to(device)
    return model(zb, mb)["z_hat"][0].float().cpu().numpy()

def seg_lengths(idx):
    return {name: int(m[idx, a:b].sum()) for name, a, b in segments}

def show_sample(idx, title):
    zt, zh, mk = z[idx].astype(np.float32), reconstruct(idx), m[idx]
    err = np.abs(zt - zh)
    fig, ax = plt.subplots(1, 2, figsize=(15, 4))
    im = ax[0].imshow(err[:, :256].T, aspect="auto", cmap="magma"); plt.colorbar(im, ax=ax[0])
    ax[0].set_title(f"{title} | |z - z_hat| (first 256 dims)"); ax[0].set_xlabel("token"); ax[0].set_ylabel("dim")
    per = ((zt - zh) ** 2).sum(-1) * mk
    ax[1].plot(per, color="crimson"); ax[1].set_title(f"{title} | per-token L2 (loss={loss[idx]:.1f})")
    for name, a, b in segments:
        ax[1].axvspan(a, b, alpha=0.05, color="blue")
    ax[1].set_xlabel("token")
    plt.tight_layout(); plt.show()
    print(f"{title} idx={idx} valid-lengths={seg_lengths(idx)} action|mean|={np.abs(act[idx]).mean():.3f}")

show_sample(int(low_idx[0]), "LOW loss")
show_sample(int(high_idx[0]), "HIGH loss")

## 8. What correlates with high loss?

Scatter loss against simple sample features (reasoning / subtask / fast valid lengths, total valid tokens, action magnitude) with Pearson `r`.

In [ ]:
seg = {name: (a, b) for name, a, b in segments}
feat = {
    "reasoning_len":  m[:, seg["reasoning"][0]:seg["reasoning"][1]].sum(1),
    "subtask_len":    m[:, seg["subtask"][0]:seg["subtask"][1]].sum(1),
    "fast_len":       m[:, seg["fast"][0]:seg["fast"][1]].sum(1),
    "total_valid":    m.sum(1),
    "action_absmean": np.abs(act).reshape(len(loss), -1).mean(1),
}
fig, axs = plt.subplots(1, len(feat), figsize=(4 * len(feat), 4))
for ax, (name, v) in zip(axs, feat.items()):
    v = v.astype(np.float64)
    ax.scatter(v, loss, s=6, alpha=0.4)
    r = np.corrcoef(v, loss)[0, 1] if v.std() > 0 else float("nan")
    ax.set_title(f"{name}\nr={r:.2f}"); ax.set_xlabel(name); ax.set_ylabel("loss")
plt.tight_layout(); plt.show()

## Notes & next steps

- `per_token_l2` is **unmasked** inside the model; here it's masked (`* mask`) so padded slots don't bias the averages.
- The shards store **embeddings only**, not token ids, so text-level decoding of high-loss reasoning/subtask isn't possible from these files alone. To read the actual language, either re-dump with token ids or cross-reference the raw GCS data (order matches `shuffle=False` dumps).
- Compare runs by pointing `RUN`/`CKPT` at a different checkpoint and re-running.
- Reference scale: the training `shuffle_baseline` (random-pair L2) — losses well below it mean the single RL token carries real information about the sequence.

## 9. View the raw input behind each loss

Requires shards dumped with `--save-metadata` (token ids → text) and `--save-images` (base_0_rgb). Each row's image/text lives in the same shard row as its embedding, so the raw input ↔ embedding ↔ loss mapping is exact (shuffle-proof).

In [ ]:
# Detokenizer (PaliGemma sentencepiece). Reserved/CoT ids (>= base vocab) are shown as <id>.
import sentencepiece
from openpi.shared import download

_tok_path = download.maybe_download("gs://big_vision/paligemma_tokenizer.model", gs={"token": "anon"})
sp = sentencepiece.SentencePieceProcessor(model_proto=open(_tok_path, "rb").read())
BASE_VOCAB = sp.get_piece_size()

def detok(ids, mask=None):
    ids = np.asarray(ids).ravel().tolist()
    if mask is not None:
        mk = np.asarray(mask).ravel().astype(bool).tolist()
        ids = [t for t, k in zip(ids, mk) if k]
    out, buf = [], []
    for t in ids:
        if 0 <= int(t) < BASE_VOCAB:
            buf.append(int(t))
        else:
            if buf: out.append(sp.decode(buf)); buf = []
            out.append(f"<{int(t)}>")
    if buf: out.append(sp.decode(buf))
    return " ".join(out).strip()

In [ ]:
def _mask_for(key, idx):
    mk = key + "_mask"
    return meta[mk][idx] if mk in meta else None

def show_input(idx):
    print(f"idx={idx}  loss={loss[idx]:.2f}  action|mean|={np.abs(act[idx]).mean():.3f}")
    for key, label in [("tokenized_prompt", "PROMPT "), ("tokenized_reasoning", "REASON "),
                       ("tokenized_subtask", "SUBTASK"), ("tokenized_fast", "FAST   ")]:
        if key in meta:
            print(f"  {label}:", detok(meta[key][idx], _mask_for(key, idx)))
    if "base_image" in meta:
        plt.figure(figsize=(4, 4)); plt.imshow(meta["base_image"][idx]); plt.axis("off")
        plt.title(f"idx {idx}  loss {loss[idx]:.1f}"); plt.show()

if not meta:
    print("No metadata in these shards. Re-dump with --save-metadata (and --save-images) to enable this.")
else:
    print("=== LOWEST-loss inputs ===")
    for i in low_idx[:3]:
        show_input(int(i))
    print("\n=== HIGHEST-loss inputs ===")
    for i in high_idx[:3]:
        show_input(int(i))